# How a System One Model Decides

**What you'll learn**
- Run decision models locally with Ollama, and see the answers change with the model
- Why asking an LLM for JSON is slow and its confidence meaningless
- How to read a probability per option straight from the next-token logits
- Check whether those probabilities can be trusted (calibration, ECE), and fix them with a temperature
- Run Laya in Python and measure it with `laya.evals`, overall and per slice
- Lab: the mistakes Laya is sure about

## Run a Decision Model Locally With Ollama
- Ollama lists them under Decision: https://ollama.com/search?c=decision
- `ollama pull tev1`: Together AI, 4B
- `ollama pull nimble`: Bespoke Labs, 9B
- `ollama pull clef-flash`: 9B
- One request: a state and typed questions in, probabilities out

In [ ]:
import pandas as pd
from typesafe_sdk import TypeSafeClient, Choice, Score, Noul

In [ ]:
tickets = pd.read_csv("data/tickets.csv")
tickets.head()

In [ ]:
client = TypeSafeClient(api_key="ollama", base_url="http://127.0.0.1:11434", timeout=300)
client

In [ ]:
QUESTIONS = {
    "department": Choice(
        instructions="Which team should handle this ticket?",
        criteria={
            "billing": "invoices, payments, refunds",
            "technical": "bugs, errors, outages",
            "account": "login, profile, users, personal data",
            "sales": "pricing, plans, quotes",
        },
    ),
    "urgency": Score(
        instructions="How urgent is this ticket?",
        criteria=["can wait", "this week", "today"],
    ),
    "refund": Noul(
        instructions="The customer explicitly asks for money back.",
    ),
}
QUESTIONS

In [ ]:
response = client.system_one(tickets["text"][0], QUESTIONS, model="tev1")
response.answers

## Compare tev1, Nimble and Clef-Flash on the Same Ticket
- Same request, only `model` changes
- Ticket 2: the whole team is locked out, gold answer `technical`

In [ ]:
ticket = tickets["text"][2]
ticket

In [ ]:
compare = []
for model_name in ["tev1", "nimble", "clef-flash"]:
    # the same request, another model
    response = client.system_one(ticket, QUESTIONS, model=model_name)
    answers = response.answers
    department = answers["department"]
    compare.append({
        "model": model_name,
        "department": department.choice,
        # its probability for that answer
        "p(department)": max(department.probabilities.values()),
        "urgency": answers["urgency"].score,
        "refund": answers["refund"].noul,
    })

compare

In [ ]:
pd.DataFrame(compare)

## Setup
- Any small instruct model works
- Qwen2.5-0.5B-Instruct: fits any GPU, and an upcoming section fine-tunes it

In [ ]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    dtype=torch.bfloat16,
    device_map="cuda",
)
model.config.num_hidden_layers

## Why Asking an LLM for a JSON Answer Falls Short
- Generate an answer token by token, then parse it
- First the steps on one ticket, then the same steps as a function

In [ ]:
prompt = (
    f"Ticket: {tickets['text'][0]}\n"
    "Classify the department as billing, technical, account or sales. "
    'Reply only with JSON: {"department": ..., "confidence": <0 to 1>}'
)
print(prompt)

In [ ]:
messages = [{"role": "user", "content": prompt}]
x = tok.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)
x = x.to("cuda")
x["input_ids"].shape

In [ ]:
# one forward pass per new token
out = model.generate(**x, max_new_tokens=40, do_sample=False)
out.shape

In [ ]:
prompt_length = x["input_ids"].shape[1]
new_tokens = out[0, prompt_length:]  # only what the model wrote
tok.decode(new_tokens, skip_special_tokens=True)

In [ ]:
@torch.no_grad()
def ask_for_json(text):
    prompt = (
        f"Ticket: {text}\n"
        "Classify the department as billing, technical, account or sales. "
        'Reply only with JSON: {"department": ..., "confidence": <0 to 1>}'
    )
    messages = [{"role": "user", "content": prompt}]
    x = tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    x = x.to("cuda")
    out = model.generate(**x, max_new_tokens=40, do_sample=False)
    prompt_length = x["input_ids"].shape[1]
    new_tokens = out[0, prompt_length:]
    return tok.decode(new_tokens, skip_special_tokens=True)

In [ ]:
start = time.perf_counter()
for text in tickets["text"][:8]:
    print(ask_for_json(text))

ms = (time.perf_counter() - start) / 8 * 1000  # milliseconds per ticket
ms

- Look at the `confidence` values the model wrote
- Look at the time per ticket in milliseconds

## Turn an Ordinary LLM Into a Decision Model With Its Logits
- Put the options behind letters, run **one** forward pass, stop
- Keep the logits of the option letters at the last position, softmax over them
- No sampling, no parsing, the answer can only be one of your options

In [ ]:
OPTIONS = {
    "billing": "invoices, payments, refunds",
    "technical": "bugs, errors, outages",
    "account": "login, profile, users, personal data",
    "sales": "pricing, plans, quotes",
}
LETTERS = ["A", "B", "C", "D"]

In [ ]:
LETTER_IDS = []
for letter in LETTERS:
    # the token id of each letter
    LETTER_IDS.append(tok.encode(letter, add_special_tokens=False)[0])
LETTER_IDS

In [ ]:
lines = ""
for letter, (name, description) in zip(LETTERS, OPTIONS.items()):
    # one option per line, behind its letter
    lines += f"{letter}. {name}: {description}\n"
print(lines)

- First the steps on one ticket, then the same steps as a function

In [ ]:
prompt = (
    f"Ticket: {tickets['text'][0]}\n\n"
    "Which department should handle this ticket?\n"
    f"{lines}\n"
    "Answer with the letter only."
)
messages = [{"role": "user", "content": prompt}]
x = tok.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)
x = x.to("cuda")
x["input_ids"].shape

In [ ]:
with torch.no_grad():
    logits = model(**x).logits  # one forward pass, no generate
logits.shape

In [ ]:
last = logits[0, -1]  # the scores for the next token, one per word in the vocabulary
last.shape

In [ ]:
letter_logits = last[LETTER_IDS].float().cpu()  # only the four letters
letter_logits

In [ ]:
probs = torch.softmax(letter_logits, dim=-1)
dict(zip(OPTIONS, probs.tolist()))

In [ ]:
@torch.no_grad()
def option_logits(text):
    prompt = (
        f"Ticket: {text}\n\n"
        "Which department should handle this ticket?\n"
        f"{lines}\n"
        "Answer with the letter only."
    )
    messages = [{"role": "user", "content": prompt}]
    x = tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    x = x.to("cuda")
    logits = model(**x).logits
    last = logits[0, -1]
    return last[LETTER_IDS].float().cpu()

## Classify All 40 Tickets and Measure Speed and Accuracy
- One forward pass per ticket, compare the time with the JSON version

In [ ]:
start = time.perf_counter()
rows = []
for text in tickets["text"]:
    rows.append(option_logits(text))  # four logits per ticket

ms = (time.perf_counter() - start) / len(tickets) * 1000  # milliseconds per ticket
print(ms)
rows[:3]

In [ ]:
L = torch.stack(rows)  # one row per ticket, one column per option
L.shape

In [ ]:
gold = []
for department in tickets["department"]:
    gold.append(list(OPTIONS).index(department))  # the right option's position
gold = torch.tensor(gold)
gold

In [ ]:
probs = torch.softmax(L, dim=-1)  # each row now sums to 1
probs[:3]

In [ ]:
conf, pred = probs.max(dim=-1)  # the top probability and its option
conf[:5], pred[:5]

In [ ]:
correct = pred == gold
correct.float().mean()

## Measure Calibration With Expected Calibration Error (ECE)
- **ECE** (expected calibration error): bucket answers by confidence, the weighted gap between how sure and how right, lower is better
- `laya.evals.ece` computes it; the same number every upcoming evaluation reports

In [ ]:
from laya.evals import ece

ece(conf.tolist(), correct.tolist())

## Calibrate the Probabilities With Temperature Scaling
- Divide the option logits by `T` before the softmax; `T > 1` softens them
- Fit `T` on labelled data by minimising the log loss, never on the test set

In [ ]:
fit = slice(0, 40, 2)  # every other ticket, to fit T
test = slice(1, 40, 2)  # the rest, to test it; both halves contain mistakes
L[fit].shape, L[test].shape

In [ ]:
losses = {}
for T in [0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.25, 1.5, 2, 3, 4]:
    # log loss with this temperature
    loss = torch.nn.functional.cross_entropy(L[fit] / T, gold[fit])
    losses[T] = loss.item()

losses

In [ ]:
best_T = min(losses, key=losses.get)  # the T with the lowest loss
best_T

In [ ]:
for T in [1, best_T]:
    probs = torch.softmax(L[test] / T, dim=-1)
    conf, pred = probs.max(dim=-1)
    correct = pred == gold[test]
    print(T, round(ece(conf.tolist(), correct.tolist()), 3))

## Run Laya Locally
- Laya is on Ollama too; upcoming sections fine-tune it, so here it runs in-process with the `laya` package
- `laya.Agent`: one checkpoint, loaded on the GPU
- Questions are plain dicts, the same shape the SDK sends

In [ ]:
import os
import laya
from laya.evals import Dataset, evaluate

In [ ]:
agent = laya.Agent("convaiinnovations/laya", device="cuda")
agent.cfg["encoder"]

In [ ]:
LAYA_QUESTIONS = {}
for qid, question in QUESTIONS.items():
    # the SDK question as a plain dict
    LAYA_QUESTIONS[qid] = question.model_dump(exclude_none=True)
LAYA_QUESTIONS

In [ ]:
result = agent.predict(tickets["text"][0], LAYA_QUESTIONS)
result["answers"]["department"]

## Laya's Three Confidence Numbers and Which One to Trust
- `confidence`: how spread out the whole distribution is (normalised entropy)
- `answer_confidence`: probability of the reported answer, the calibrated one
- `action.act_probability`: a separate head that chooses act or escalate

In [ ]:
answer = result["answers"]["department"]
answer["confidence"], answer["answer_confidence"], answer["action"]["act_probability"]

## Prepare a Labelled Test Set for Laya
- One JSON line per ticket: `state`, `questions`, and the `expected` answer per question
- `choice` expects the option name, `score` the level, `noul` true or false
- `tags`: labels to slice the results by

In [ ]:
rows = []
for ticket in tickets.to_dict("records"):  # one ticket at a time
    rows.append({
        "state": ticket["text"],
        "questions": LAYA_QUESTIONS,  # the same three questions for every ticket
        "expected": {
            "department": ticket["department"],
            "urgency": ticket["urgency"],
            "refund": ticket["refund"],
        },
        "tags": [ticket["department"]],  # slice the results by department
    })

rows[0]

In [ ]:
table = pd.DataFrame(rows)
table.head()

In [ ]:
os.makedirs("runs", exist_ok=True)
table.to_json("runs/tickets.jsonl", orient="records", lines=True)

with open("runs/tickets.jsonl") as f:
    first_line = f.readline()  # one row, exactly as evaluate reads it
print(first_line)

## Measure Accuracy and Calibration With laya.evals
- Every row through the agent, each answer scored against `expected`
- `choice` and `noul`: accuracy; `score`: mean absolute error in levels
- ECE and Brier read `answer_confidence`

In [ ]:
data = Dataset.from_jsonl("runs/tickets.jsonl")
len(data)

In [ ]:
report = evaluate(agent, data)  # every ticket through Laya
report.overall.keys()

In [ ]:
overall = pd.Series(report.overall)
overall[["choice_accuracy", "noul_accuracy", "score_mae", "ece", "brier"]]

## Accuracy per Question and per Department
- `qid`: one row per question
- `tag`: one row per tag, here the gold department

In [ ]:
by_question = pd.DataFrame.from_dict(report.slices["qid"], orient="index")
by_question[["choice_accuracy", "noul_accuracy", "score_mae", "ece"]]

In [ ]:
by_department = pd.DataFrame.from_dict(report.slices["tag"], orient="index")
by_department[["choice_accuracy", "noul_accuracy", "ece"]]

## Lab: Find the Mistakes Laya Made With High Confidence
- `report.cases`: one record per answer, with `expected`, the `answer` and `correct`
- A confidence gate catches the unsure mistakes, the sure ones get through
- Look at the top of the list

In [ ]:
report.cases[0]

In [ ]:
cases = pd.json_normalize(report.cases)  # one row per answer, nested fields as columns
cases.columns

In [ ]:
wrong = cases[cases["correct"] == False]  # only the wrong answers
len(wrong)

In [ ]:
wrong = wrong.sort_values("confidence", ascending=False)  # the surest first
wrong[["qid", "expected", "answer.choice", "answer.noul", "confidence"]]

## Cheat Sheet
| Step | Generate JSON | Read logits |
|---|---|---|
| Forward passes | one per output token | one |
| Output | free text you must parse | a probability per option |
| Invalid label possible | yes | no |
| Confidence | a number it wrote | measured, then calibrated with a temperature |

| Need | Code |
|---|---|
| Ollama model | `TypeSafeClient(api_key="ollama", base_url="http://127.0.0.1:11434").system_one(state, questions, model=...)` |
| Laya in Python | `laya.Agent("convaiinnovations/laya", device="cuda").predict(state, questions)` |
| Gate on | `answer["answer_confidence"]`, not `confidence` |
| Labelled rows | JSONL with `state`, `questions`, `expected`, `tags` |
| Score | `evaluate(agent, Dataset.from_jsonl(path))` |
| Results | `report.overall`, `report.slices["qid"]`, `report.slices["tag"]`, `report.cases` |